In [1]:
new_run_id = None
old_run_id = None
name = '2024-12-11 Migration: Test 1'
description = None
user = 'karen.king@autointel.io'
objective_function = 'Maximize Bank Balance'
duration = 87
overhead_per_period = 1000000.0
mip_gap = 0.001
min_crossover_point = None
is_infeasible = "No"
infeasibility_slack_approach = "Balances"
prohibit_money_same_period = "Yes"
enable_net_new_constraint = "No"
net_new_window = None
net_new_max_percentage = None
infeasibility_slack_objective = "Combine with Main Objective"
enable_relative_deployments = "No"
relative_deployments_relative_tolerance = 0.0
relative_deployments_absolute_tolerance = 0.0
deviation_minimalization_mode = "Max"
optimal_objective_value = None
solver_params = None
skip_slack_model = "Yes"
relative_deployments_max_deviation = None
copy_parameters = False  # if True, hiver_optimizer parameters are copied invariably from old_run_id to new_run_id

StatementMeta(, 06df5b2e-0705-453a-ab4c-8cfe5da34a3d, 3, Finished, Available, Finished, False)

In [1]:
import calendar
import datetime
import logging
import traceback
import uuid
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType, BooleanType

StatementMeta(, 5f5f0b8f-4291-493e-9803-f66529907cc4, 3, Finished, Available, Finished)

In [2]:
#old_run_id = '8d58e0f4-165c-4983-b4b4-4471c05614c9'

StatementMeta(, 5f5f0b8f-4291-493e-9803-f66529907cc4, 4, Finished, Available, Finished)

In [2]:
def get_derived_parameters(old_run_id: str, duration: int):
    '''
    Return start_balance, start_period, end_period
    '''

    # Get today's date
    today = datetime.date.today()
    first_date_of_this_month = today.replace(day=1)

    # TEMP: `month_end_actuals` is not updated.  Get last date in table
    sql = f"""
    SELECT Period
    FROM month_end_actuals
    WHERE Portfolio = 'All'
    AND Period < '{first_date_of_this_month.strftime("%Y-%m-%d")}'
    ORDER BY Period DESC
    LIMIT 1
    """

    last_date_of_previous_month = spark.sql(sql).toPandas().Period.iloc[0]

    # Calculate last date of this month & previous month
    first_date_of_this_month = last_date_of_previous_month + datetime.timedelta(days=1)
    _, last_day_of_this_month = calendar.monthrange(first_date_of_this_month.year, first_date_of_this_month.month)

    last_date_of_this_month = first_date_of_this_month.replace(day=last_day_of_this_month)

    last_date_of_this_month = last_date_of_this_month.strftime("%Y-%m-%d")
    last_date_of_previous_month = last_date_of_previous_month.strftime("%Y-%m-%d")

    # StartingBalance = Balance at last day of previous month
    sql = f"""
    SELECT Balance
    FROM month_end_actuals
    WHERE Portfolio = 'All'
    AND Period = '{last_date_of_previous_month}'
    """

    start_balance = spark.sql(sql).toPandas().Balance.iloc[0]
    print(start_balance)

    # StartPeriod = PeriodID for this month (taken from data for old RunID)
    sql = f"""
    SELECT PeriodID
    FROM input_time_periods
    WHERE PeriodLabel = '{first_date_of_this_month}'
    AND RunID = '{old_run_id}'
    """

    #start_period = spark.sql(sql).toPandas().PeriodID.iloc[0]
    # Temp  - Only for weekly
    sql = f"""
    SELECT StartPeriod
    FROM input_parameters
    WHERE RunID = '{old_run_id}'
    """
    start_period = spark.sql(sql).toPandas().StartPeriod.iloc[0]
    print(start_period)

    # EndPeriod = StartPeriod + Duration
    end_period = start_period + duration
    #end_period 
    print(end_period)

    # Truncate to last period within range that has data
    sql = f"""
    SELECT MIN(MaxPeriodID) as EndPeriod
    FROM (
        SELECT MAX(PeriodID) as MaxPeriodID
        FROM input_customer_capacities
        WHERE PeriodID BETWEEN {start_period} AND {end_period}
        AND RunID = '{old_run_id}'
        UNION
        SELECT MAX(PeriodID) as MaxPeriodID
        FROM input_performance_curves
        WHERE PeriodID BETWEEN {start_period} AND {end_period}
        AND RunID = '{old_run_id}'
        UNION
        SELECT MAX(PeriodID) as MaxPeriodID
        FROM input_periods_config
        WHERE PeriodID BETWEEN {start_period} AND {end_period}
        AND RunID = '{old_run_id}'
        UNION
        SELECT MAX(ToPeriodID) as MaxPeriodID
        FROM input_portfolio_deployments
        WHERE ToPeriodID BETWEEN {start_period} AND {end_period}
        AND RunID = '{old_run_id}'
        UNION
        SELECT MAX(PeriodID) as MaxPeriodID
        FROM input_repeats_distribution
        WHERE PeriodID BETWEEN {start_period} AND {end_period}
        AND RunID = '{old_run_id}'
        UNION
        SELECT MAX(PeriodID) as MaxPeriodID
        FROM input_time_periods
        WHERE PeriodID BETWEEN {start_period} AND {end_period}
        AND RunID = '{old_run_id}'
    )
    """

    end_period = spark.sql(sql).toPandas().EndPeriod.iloc[0]

    return float(start_balance), int(start_period), int(end_period)

StatementMeta(, 42d9618f-f191-49e2-ab85-5db53f023e52, 4, Finished, Available, Finished)

In [17]:
# old_run_id = '8d58e0f4-165c-4983-b4b4-4471c05614c9'
# duration = 1351
# start_balance, start_period, end_period = get_derived_parameters(old_run_id, duration)


StatementMeta(, 5f5f0b8f-4291-493e-9803-f66529907cc4, 19, Finished, Available, Finished)

11521822.74
400
1751


In [ ]:
def insert_run(new_run_id, name, description, user):
    # Set Defaults
    user = user or 'fedo@hivefs.com'
    name = name or f"Run {new_run_id}"
    description = description or ""
    status = 'Running'

    # Create a DF
    data = [(new_run_id, name, status, description, user, '', '')]
    columns = ["RunID", "Name", "Status", "Notes", "User", "ErrorMessage", "ModelVersion"]

    # Convert to Spark & Add Date
    df = spark.createDataFrame(data, schema=columns).withColumn("Date", F.current_timestamp())

    # Insert Into runs
    df.select("RunID", "Date", "Name", "Status", "Notes", "User", "ErrorMessage", "ModelVersion").write.insertInto("runs", overwrite=False)


In [ ]:
def insert_parameters(
    new_run_id,
    objective_function,
    start_balance,
    start_period,
    end_period,
    mip_gap,
    min_crossover_point,
    is_infeasible,
    infeasibility_slack_approach,
    prohibit_money_same_period,
    enable_net_new_constraint,
    net_new_window,
    net_new_max_percentage,
    infeasibility_slack_objective,
    enable_relative_deployments,
    relative_deployments_relative_tolerance,
    relative_deployments_absolute_tolerance,
    deviation_minimalization_mode,
    optimal_objective_value,
    solver_params,
    skip_slack_model,
    relative_deployments_max_deviation,
):

    # Create a DF
    min_crossover = int(min_crossover_point) if min_crossover_point else None

    data = [(
        new_run_id,
        start_balance,
        start_period,
        end_period,
        mip_gap,
        objective_function,
        min_crossover,
        is_infeasible,
        infeasibility_slack_approach,
        prohibit_money_same_period,
        enable_net_new_constraint,
        net_new_window,
        net_new_max_percentage,
        infeasibility_slack_objective,
        enable_relative_deployments,
        relative_deployments_relative_tolerance,
        relative_deployments_absolute_tolerance,
        deviation_minimalization_mode,
        optimal_objective_value,
        solver_params,
        skip_slack_model,
        relative_deployments_max_deviation,
    )]

    schema = StructType([
        StructField("RunID", StringType(), True),
        StructField("StartBalance", DoubleType(), True),
        StructField("StartPeriod", IntegerType(), True),
        StructField("EndPeriod", IntegerType(), True),
        StructField("MIPGap", DoubleType(), True),
        StructField("MainObjectiveFunction", StringType(), True),
        StructField("MinCrossoverPoint", IntegerType(), True),
        StructField("IsInfeasible", StringType(), True),
        StructField("InfeasibilitySlackApproach", StringType(), True),
        StructField("ProhibitMoneySamePeriod", StringType(), True),
        StructField("EnableNetNewConstraint", StringType(), True),
        StructField("NetNewWindow", IntegerType(), True),
        StructField("NetNewMaxPercentage", DoubleType(), True),
        StructField("InfeasibilitySlackObjective", StringType(), True),
        StructField("EnableRelativeDeployments", StringType(), True),
        StructField("RelativeDeploymentsRelTol", DoubleType(), True),
        StructField("RelativeDeploymentsAbsTol", DoubleType(), True),
        StructField("DeviationMinimizationMode", StringType(), True),
        StructField("OptimalObjectiveValue", DoubleType(), True),
        StructField("SolverParams", StringType(), True),
        StructField("SkipSlackModel", StringType(), True),
        StructField("RelativeDeploymentsMaxDeviation", StringType(), True),
    ])

    # Convert to Spark
    df = spark.createDataFrame(data, schema=schema)

    # Insert
    df.write.insertInto("input_parameters", overwrite=False)

In [ ]:
def copy_data(old_run_id, new_run_id, copy_parameters: bool):
    table_names = spark.sql("SHOW TABLES").toPandas()

    for table_name in table_names.tableName.unique():
        if table_name.startswith('input_'):
            
            # if copy_parameters=True, input_parameters is not skipped
            if ((not copy_parameters) and (table_name == 'input_parameters')) or table_name.endswith('_test'):
                continue

            columns = spark.sql(f"DESCRIBE {table_name}").toPandas()
            column_names = columns.col_name.unique().tolist()

            if 'RunID' not in column_names:
                print(f"Skipping table: {table_name}")
                continue

            column_clause = ", ".join(c for c in column_names if c != 'RunID')

            sql = f"""
                INSERT INTO {table_name}
                (RunID, {column_clause})
                SELECT '{new_run_id}', {column_clause}
                FROM {table_name}
                WHERE RunID = '{old_run_id}'
            """

            print("***********************")
            print(table_name)
            print("***********************")
            print(sql)
            print()
            spark.sql(sql)

In [ ]:
def create_run(
    old_run_id,
    new_run_id,
    name,
    description,
    user,
    objective_function,
    duration,
    mip_gap,
    min_crossover_point,
    is_infeasible,
    infeasibility_slack_approach,
    prohibit_money_same_period,
    enable_net_new_constraint,
    net_new_window,
    net_new_max_percentage,
    infeasibility_slack_objective,
    enable_relative_deployments,
    relative_deployments_relative_tolerance,
    relative_deployments_absolute_tolerance,
    deviation_minimalization_mode,
    optimal_objective_value,
    solver_params,
    skip_slack_model,
    relative_deployments_max_deviation,
    copy_parameters
):

    status = 'Success'
    error_message = ''

    try:
        if old_run_id is None:
            raise ValueError("A value is required for `old_run_id`")

        if new_run_id is None:
            new_run_id = str(uuid.uuid4())
        
        insert_run(new_run_id, name, description, user)
        copy_data(old_run_id, new_run_id, copy_parameters)

        if not copy_parameters:
            start_balance, start_period, end_period = get_derived_parameters(old_run_id, duration)
            insert_parameters(
                new_run_id, 
                objective_function, 
                start_balance, 
                start_period, 
                end_period, 
                mip_gap, 
                min_crossover_point, 
                is_infeasible, 
                infeasibility_slack_approach, 
                prohibit_money_same_period,
                enable_net_new_constraint,
                net_new_window,
                net_new_max_percentage,
                infeasibility_slack_objective,
                enable_relative_deployments,
                relative_deployments_relative_tolerance,
                relative_deployments_absolute_tolerance,
                deviation_minimalization_mode,
                optimal_objective_value,
                solver_params,
                skip_slack_model,
                relative_deployments_max_deviation,
            )
    except Exception as e:
        status = 'Failed'
        error_message = traceback.format_exc()
        logging.error(f"An error occurred: {e}")
        logging.error(error_message)

    return {'new_run_id': new_run_id, 'status': status, 'error_message': error_message}

In [ ]:
# Capture any parameters being passed via `mssparkutils`
try:
    new_run_id = mssparkutils.notebook._get_input("new_run_id", new_run_id)
    old_run_id = mssparkutils.notebook._get_input("old_run_id", old_run_id)
    name = mssparkutils.notebook._get_input("name", name)
    description = mssparkutils.notebook._get_input("description", description)
    user = mssparkutils.notebook._get_input("user", user)
    objective_function = mssparkutils.notebook._get_input("objective_function", objective_function)
    duration = mssparkutils.notebook._get_input("duration", duration)
    mip_gap = mssparkutils.notebook._get_input("mip_gap", mip_gap)
    min_crossover_point = mssparkutils.notebook._get_input("min_crossover_point", min_crossover_point)
    is_infeasible = mssparkutils.notebook._get_input("is_infeasible", is_infeasible)
    infeasibility_slack_approach = mssparkutils.notebook._get_input("infeasibility_slack_approach", infeasibility_slack_approach)
    prohibit_money_same_period = mssparkutils.notebook._get_input("prohibit_money_same_period", prohibit_money_same_period)
    enable_net_new_constraint = mssparkutils.notebook._get_input("enable_net_new_constraint", enable_net_new_constraint)
    net_new_window = mssparkutils.notebook._get_input("net_new_window", net_new_window)
    net_new_max_percentage = mssparkutils.notebook._get_input("net_new_max_percentage", net_new_max_percentage)
    infeasibility_slack_objective = mssparkutils.notebook._get_input("infeasibility_slack_objective", infeasibility_slack_objective)
    enable_relative_deployments = mssparkutils.notebook._get_input("enable_relative_deployments", enable_relative_deployments)
    relative_deployments_relative_tolerance = mssparkutils.notebook._get_input("relative_deployments_relative_tolerance", relative_deployments_relative_tolerance)
    relative_deployments_absolute_tolerance = mssparkutils.notebook._get_input("relative_deployments_absolute_tolerance", relative_deployments_absolute_tolerance)
    deviation_minimalization_mode = mssparkutils.notebook._get_input("deviation_minimalization_mode", deviation_minimalization_mode)
    optimal_objective_value = mssparkutils.notebook._get_input("optimal_objective_value", optimal_objective_value)
    solver_params = mssparkutils.notebook._get_input("solver_params", solver_params)
    skip_slack_model = mssparkutils.notebook._get_input("skip_slack_model", skip_slack_model)
    relative_deployments_max_deviation = mssparkutils.notebook._get_input("relative_deployments_max_deviation", relative_deployments_max_deviation)
    copy_parameters = mssparkutils.notebook._get_input("copy_parameters", copy_parameters)
except:
    # Skip if parameters not being passed via `mssparkutils`
    pass

In [ ]:
result = create_run(
    old_run_id,
    new_run_id,
    name,
    description,
    user,
    objective_function,
    duration,
    mip_gap,
    min_crossover_point,
    is_infeasible,
    infeasibility_slack_approach,
    prohibit_money_same_period,
    enable_net_new_constraint,
    net_new_window,
    net_new_max_percentage,
    infeasibility_slack_objective,
    enable_relative_deployments,
    relative_deployments_relative_tolerance,
    relative_deployments_absolute_tolerance,
    deviation_minimalization_mode,
    optimal_objective_value,
    solver_params,
    skip_slack_model,
    relative_deployments_max_deviation,
    copy_parameters
)
mssparkutils.notebook.exit(str(result))